In [1]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
# DATASET
class CodeDataset(Dataset):
    def __init__(self, bin_file, block_size=1024):
        self.block_size = block_size
        self.data = np.fromfile(bin_file, dtype=np.uint16)
        self.n_samples = len(self.data) - block_size

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.block_size]
        y = self.data[idx + 1: idx + self.block_size + 1]

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long)
        )

In [3]:
# MODEL
class GPTEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        token_emb = self.token_embedding(x)
        positions = torch.arange(T, device=x.device)
        pos_emb = self.position_embedding(positions)
        return self.dropout(token_emb + pos_emb)


In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        mask = self.mask[:T, :T]
        mask = mask.unsqueeze(0).unsqueeze(0)
        att = att.masked_fill(mask == 0, float("-inf"))

        att = torch.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.resid_dropout(self.out(out))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(0.1),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, block_size)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = FeedForward(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size=50000, embed_dim=512, block_size=1024, num_heads=8, num_layers=6):
        super().__init__()

        self.embedding = GPTEmbedding(vocab_size, embed_dim, block_size)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, block_size)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.head.weight = self.embedding.token_embedding.weight

    def forward(self, x, targets=None):
        x = self.embedding(x)
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)
            loss = nn.CrossEntropyLoss()(logits, targets)

        return logits, loss

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()

    total_loss = 0
    total_correct = 0
    total_tokens = 0
    num_batches = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits, loss = model(x, y)

            total_loss += loss.item()
            num_batches += 1

            preds = logits.argmax(dim=1)
            total_correct += (preds == y.view(-1)).sum().item()
            total_tokens += y.numel()

    avg_loss = total_loss / num_batches
    perplexity = math.exp(avg_loss)
    token_accuracy = total_correct / total_tokens

    return avg_loss, perplexity, token_accuracy

In [ ]:
# MAIN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

TRAIN_BIN = os.path.join(BASE_DIR, "Data", "tokenized", "train.bin")
VAL_BIN = os.path.join(BASE_DIR, "Data", "tokenized", "val.bin")

train_dataset = CodeDataset(TRAIN_BIN, block_size=1024)
val_dataset = CodeDataset(VAL_BIN, block_size=1024)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

checkpoints = {
    "5000": os.path.join("src", "model", "model_step_5000.pt"),
    "10000": os.path.join("src", "model", "model_step_10000.pt")
}

results = {}

for name, path in checkpoints.items():
    print(f"\nEvaluating model {name}...")

    model = GPT().to(device)

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model"])

    # TRAIN METRICS
    train_loss, train_ppl, train_acc = evaluate_model(
        model,
        train_loader,
        device
    )

    # VAL METRICS
    val_loss, val_ppl, val_acc = evaluate_model(
        model,
        val_loader,
        device
    )

    gap = val_loss - train_loss

    results[name] = {
        "train_loss": train_loss,
        "train_ppl": train_ppl,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_ppl": val_ppl,
        "val_acc": val_acc,
        "gap": gap
    }

    print(f"\nMODEL {name}")
    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Perplexity: {train_ppl:.4f}")
    print(f"Train Accuracy  : {train_acc:.4%}")

    print(f"Val Loss        : {val_loss:.4f}")
    print(f"Val Perplexity  : {val_ppl:.4f}")
    print(f"Val Accuracy    : {val_acc:.4%}")

    print(f"Overfit Gap     : {gap:.4f}")

# BEST MODEL SELECTION
best_model = min(
    results.items(),
    key=lambda x: x[1]["val_loss"]
)

print("\n" + "=" * 60)
print(f"BEST MODEL (lowest validation loss): model_step_{best_model[0]}.pt")
print("=" * 60)

# OVERFITTING CHECK
for name, metrics in results.items():
    if metrics["gap"] > 0.5:
        print(f"WARNING: model_step_{name}.pt may be overfitting.")
    else:
        print(f"model_step_{name}.pt shows acceptable generalization.")

In [ ]:

# ======================================================================
# EVALUATING SUBSET SIZE: 5000
# ======================================================================

# Evaluating model 5000 on subset 5000
# Train 5000 (5000): 100%|██████████| 313/313 [03:48<00:00,  1.37it/s]
# Val 5000 (5000): 100%|██████████| 313/313 [03:47<00:00,  1.38it/s]

# --------------------------------------------------
# MODEL 5000 | SUBSET 5000
# --------------------------------------------------
# Train Loss      : 1.9316
# Train Perplexity: 6.9005
# Train Accuracy  : 57.8628%
# Val Loss        : 2.7412
# Val Perplexity  : 15.5052
# Val Accuracy    : 53.3186%
# Overfit Gap     : 0.8096

# Evaluating model 10000 on subset 5000
# Train 10000 (5000): 100%|██████████| 313/313 [03:47<00:00,  1.38it/s]
# Val 10000 (5000): 100%|██████████| 313/313 [03:47<00:00,  1.38it/s]

# --------------------------------------------------
# MODEL 10000 | SUBSET 5000
# --------------------------------------------------
# Train Loss      : 1.2030
# Train Perplexity: 3.3300
# Train Accuracy  : 70.3315%
# Val Loss        : 2.5356
# Val Perplexity  : 12.6235
# Val Accuracy    : 58.7741%
# Overfit Gap     : 1.3326

# ============================================================
# BEST MODEL for subset 5000: model_step_10000.pt
# ============================================================

# ======================================================================
# EVALUATING SUBSET SIZE: 10000
# ======================================================================

# Evaluating model 5000 on subset 10000
# Train 5000 (10000): 100%|██████████| 625/625 [07:34<00:00,  1.37it/s]
# Val 5000 (10000): 100%|██████████| 625/625 [07:34<00:00,  1.37it/s]

# --------------------------------------------------
# MODEL 5000 | SUBSET 10000
# --------------------------------------------------
# Train Loss      : 1.7352
# Train Perplexity: 5.6701
# Train Accuracy  : 61.7836%
# Val Loss        : 2.9118
# Val Perplexity  : 18.3901
# Val Accuracy    : 52.3594%
# Overfit Gap     : 1.1766

# Evaluating model 10000 on subset 10000
# Train 10000 (10000): 100%|██████████| 625/625 [07:34<00:00,  1.38it/s]
# Val 10000 (10000): 100%|██████████| 625/625 [07:34<00:00,  1.38it/s]

# --------------------------------------------------
# MODEL 10000 | SUBSET 10000
# --------------------------------------------------
# Train Loss      : 1.0853
# Train Perplexity: 2.9603
# Train Accuracy  : 73.0954%
# Val Loss        : 2.8329
# Val Perplexity  : 16.9938
# Val Accuracy    : 56.8388%
# Overfit Gap     : 1.7476

# ============================================================
# BEST MODEL for subset 10000: model_step_10000.pt
# ============================================================

# ======================================================================
# EVALUATING SUBSET SIZE: 15000
# ======================================================================

# Evaluating model 5000 on subset 15000
# Train 5000 (15000): 100%|██████████| 938/938 [11:21<00:00,  1.38it/s]
# Val 5000 (15000): 100%|██████████| 938/938 [11:20<00:00,  1.38it/s]

# --------------------------------------------------
# MODEL 5000 | SUBSET 15000
# --------------------------------------------------
# Train Loss      : 1.6852
# Train Perplexity: 5.3934
# Train Accuracy  : 63.1536%
# Val Loss        : 2.8679
# Val Perplexity  : 17.6005
# Val Accuracy    : 52.4483%
# Overfit Gap     : 1.1828

# Evaluating model 10000 on subset 15000
# Train 10000 (15000): 100%|██████████| 938/938 [11:20<00:00,  1.38it/s]
# Val 10000 (15000): 100%|██████████| 938/938 [11:21<00:00,  1.38it/s]

# --------------------------------------------------
# MODEL 10000 | SUBSET 15000
# --------------------------------------------------
# Train Loss      : 0.9823
# Train Perplexity: 2.6705
# Train Accuracy  : 75.4941%
# Val Loss        : 2.8122
# Val Perplexity  : 16.6473
# Val Accuracy    : 56.6064%
# Overfit Gap     : 1.8300

# ============================================================
# BEST MODEL for subset 15000: model_step_10000.pt
# ============================================================

# ======================================================================
# EVALUATING SUBSET SIZE: 20000
# ======================================================================

# Evaluating model 5000 on subset 20000
# Train 5000 (20000): 100%|██████████| 1250/1250 [15:05<00:00,  1.38it/s]
# Val 5000 (20000): 100%|██████████| 1250/1250 [15:07<00:00,  1.38it/s]

# --------------------------------------------------
# MODEL 5000 | SUBSET 20000
# --------------------------------------------------
# Train Loss      : 1.6886
# Train Perplexity: 5.4119
# Train Accuracy  : 62.9114%
# Val Loss        : 2.8017
# Val Perplexity  : 16.4729
# Val Accuracy    : 52.6803%
# Overfit Gap     : 1.1131

# Evaluating model 10000 on subset 20000
# Train 10000 (20000): 100%|██████████| 1250/1250 [15:07<00:00,  1.38it/s]
# Val 10000 (20000): 100%|██████████| 1250/1250 [15:07<00:00,  1.38it/s]
# --------------------------------------------------
# MODEL 10000 | SUBSET 20000
# --------------------------------------------------
# Train Loss      : 0.9775
# Train Perplexity: 2.6578
# Train Accuracy  : 75.3628%
# Val Loss        : 2.7676
# Val Perplexity  : 15.9202
# Val Accuracy    : 56.4813%
# Overfit Gap     : 1.7901

# ============================================================
# BEST MODEL for subset 20000: model_step_10000.pt
# ============================================================



In [ ]:
# model 10000 is selected